# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (`data.csv`), dictionary in `columns.csv`.

Plan: inspect data → check for leakage → build a leakage-free model → evaluate with a **time-based** split (train on past, test on future) plus grouped CV → compare with a baseline.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width',150)
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
df['snapshot_month']=pd.PeriodIndex(df['snapshot_month'],freq='M')
print(df.shape, 'accounts:',df.account_ref.nunique()); df.head()

In [ ]:
print(df.isna().sum().sum(),'missing;',df.duplicated().sum(),'dup rows;',df.duplicated(['account_ref','snapshot_month']).sum(),'dup account-months')
print('churn rate: %.3f'%df.churn_next_month.mean())
print(df.describe().T)
print(df.groupby('plan').churn_next_month.agg(['mean','size']))

## Leakage check: `lifetime_months`
The dictionary says it is the *total* months an account was active, taken from a CRM export dated 2026-01-05 — i.e. **after** every snapshot. That is information from the future. If an account churns the month after the snapshot, its lifetime should equal its tenure at the snapshot.

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('churners with gap==0: %.1f%%   non-churners with gap==0: %.1f%%'%(
  100*(df[df.churn_next_month==1].gap==0).mean(),100*(df[df.churn_next_month==0].gap==0).mean()))

Every churner has `lifetime_months == tenure_months` and no non-churner does. The column encodes the label itself and **would not be available at prediction time**, so it is excluded from the real model. We still score it below to show how misleading it is.

In [ ]:
df=df.sort_values('snapshot_month').reset_index(drop=True)
y=df.churn_next_month
num=['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
cat=['plan']
cut=pd.Period('2025-07','M')           # train on earlier snapshots, test on the last 5 months
tr=(df.snapshot_month<cut).values; te=~tr
print('train rows',tr.sum(),'test rows',te.sum(),'| test rows whose account is also in train: %.0f%%'%(100*df[te].account_ref.isin(df[tr].account_ref).mean()))
print('test base rate: %.3f'%y[te].mean())
def pipe(model,feats_num):
    pre=ColumnTransformer([('n',StandardScaler(),feats_num),('c',OneHotEncoder(),cat)])
    return make_pipeline(pre,model)
def evaluate(name,p,feats):
    p.fit(df.loc[tr,feats],y[tr]); s=p.predict_proba(df.loc[te,feats])[:,1]
    return dict(model=name,ROC_AUC=roc_auc_score(y[te],s),PR_AUC=average_precision_score(y[te],s))

In [ ]:
F=num+cat; Fl=F+['lifetime_months']
rows=[evaluate('baseline (prior)',pipe(DummyClassifier(),num),F),
 evaluate('logistic',pipe(LogisticRegression(max_iter=1000),num),F),
 evaluate('grad. boosting',pipe(HistGradientBoostingClassifier(random_state=0),num),F),
 evaluate('LEAKY: grad. boosting + lifetime_months',pipe(HistGradientBoostingClassifier(random_state=0),num+['lifetime_months']),Fl)]
res=pd.DataFrame(rows).set_index('model').round(3); res

The leaky model's near-perfect score is meaningless for deployment. The honest comparison is the first three rows.

In [ ]:
# robustness: grouped CV (no account in both train and validation folds)
gk=GroupKFold(5)
for name,m in [('logistic',pipe(LogisticRegression(max_iter=1000),num)),('grad. boosting',pipe(HistGradientBoostingClassifier(random_state=0),num))]:
    s=cross_val_predict(m,df[F],y,groups=df.account_ref,cv=gk,method='predict_proba')[:,1]
    print(name,'grouped-CV ROC AUC %.3f  PR AUC %.3f (base rate %.3f)'%(roc_auc_score(y,s),average_precision_score(y,s),y.mean()))

In [ ]:
# what drives it?
m=pipe(HistGradientBoostingClassifier(random_state=0),num).fit(df.loc[tr,F],y[tr])
pi=permutation_importance(m,df.loc[te,F],y[te],scoring='roc_auc',n_repeats=10,random_state=0)
pd.Series(pi.importances_mean,index=F).sort_values().plot.barh(title='Permutation importance (drop in test ROC AUC)'); plt.show()
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,c in zip(ax,['tenure_months','usage_hours','support_tickets']):
    q=pd.qcut(df[c],6,duplicates='drop') if c!='support_tickets' else df[c].clip(upper=4)
    df.groupby(q,observed=True).churn_next_month.mean().plot(marker='o',ax=a,title='churn rate by '+c); a.set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
# lift by score decile on the test period
s=m.predict_proba(df.loc[te,F])[:,1]; t=pd.DataFrame({'s':s,'y':y[te].values})
t['decile']=pd.qcut(t.s.rank(method='first'),10,labels=False)
print(t.groupby('decile').y.agg(['mean','size']).round(3).iloc[::-1])
print(res.to_string())
print('Top-decile churn rate: %.1f%% vs base %.1f%%'%(100*t[t.decile==9].y.mean(),100*t.y.mean()))

## Conclusions
See the notebook outputs and the summary in the accompanying message.

* **`lifetime_months` is leakage** (post-snapshot CRM export; equals tenure for every churner) and is excluded. It should not be used in any production model.
* Honest performance is in the time-split table and the grouped-CV output, to be read against the baseline (PR AUC of the prior ≈ the churn base rate).
* Caveats: ~15% churn rate, only 23 months of history, repeated accounts across rows (hence grouped CV), no price-change / contract / segment features.